# Intervention axes: three navigation policies, three estimands

This notebook fits three policy questions with targeted maximum likelihood estimation (TMLE). Each
policy defines its own estimand, and the result tables look alike. Each step shows its code, its
output, and what the output tells you. The
[variations](../technical-reference/point-treatment-tmle.md#variations) section gives the clever
covariate of each policy.

## The applied question

The program office has three proposals for next year's navigation standard.

| proposal | what it changes | which axis |
| --- | --- | --- |
| offer navigation only when a baseline discharge-risk screen flags the patient | who gets the standard offer, as a function of recorded baseline information | a known regime |
| raise the navigation intensity assigned at discharge, but never above a declared capacity | how much of a continuous exposure each patient receives | a modified treatment policy |
| change each patient's assignment probability from its current odds to twice those odds | the assignment mechanism itself, not one fixed assignment | an incremental propensity-score intervention |

None of the three proposals is the average treatment effect (ATE). The regime fit also reports the
ATE, as offer to all against offer to none, for comparison. The regime and the incremental policy
keep the [shared study design](index.md#the-shared-study-design). The intensity policy uses its own
continuous synthetic law. The office fixes the rule, the cap, and the odds multiplier before
fitting.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| write one protocol for each policy question | Steps 3, 8, and 12 |
| fit a known regime, and read the support of each plan | Steps 4 to 6 |
| fit a modified treatment policy on a continuous exposure | Steps 7 to 9 |
| explain why the cap is part of the question | Step 10, the failure mode |
| read the support of each shift, and see practical positivity strain | Step 11 |
| fit an incremental intervention, and say why it depends on the treatment mechanism | Steps 12 and 13 |
| tell three alike tables apart, and read their assessments | Step 14 |
| use the sensitivity analysis that applies | Step 15 |

## Why these are three estimands

| your situation | what the axis buys | what it costs |
| --- | --- | --- |
| the policy is a rule on recorded variables | a mean under a plan fixed before fitting. It needs support only for the arm the rule assigns | the rule is part of the estimand. Two rules are two parameters |
| the exposure is continuous | the mean under a shift of the observed intensity, with the achievable maximum declared | positivity becomes a statement about the conditional density, and each shift has its own support |
| you can implement a stochastic assignment rule | a tilt of the observed mechanism | the mechanism defines the target, so its inference depends on a good estimate of the mechanism |

The three results differ because the counterfactual worlds differ. The estimators do not disagree.

The table below defines the terms this notebook uses most. Each link goes to the canonical
definition.

| term | plain meaning | definition |
| --- | --- | --- |
| estimand | the number the question asks for, written before any model is chosen | [estimands](../user-guide/estimands.md) |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q, and the treatment mechanism g or the treatment density | [point-treatment TMLE](../technical-reference/point-treatment-tmle.md) |
| targeting | a small update to Q, weighted by a clever covariate built from g, that removes first-order bias | [targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) |
| clever covariate | the weight each row carries in targeting. Each axis builds its own | [variations](../technical-reference/point-treatment-tmle.md#variations) |
| positivity | every kind of patient has some chance of the treatment the policy asks for | [diagnostics](../user-guide/results-assessment.md#diagnostics) |
| effective sample size | (the sum of the weights)² divided by the sum of the squared weights. It is the number of equally weighted rows that give the same precision. When a few rows carry large weights, it falls far below n. The reports call it the Kish effective sample size | [support reports](../technical-reference/point-treatment-tmle.md#support-reports) |
| score load | the effective sample size of the absolute weights in one targeting equation, with the truncation and observation weights the fit used. It describes one equation, not the whole estimator | [support reports](../technical-reference/point-treatment-tmle.md#support-reports) |

## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
import warnings
from dataclasses import replace

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The regime and the incremental policy use the synthetic law of
[point-treatment TMLE](point-treatment-tmle.ipynb) with a different seed. `navigation_data` draws
that law, `make_nonlinear_bounded`, under the program's column names. The score is a share of the
maximum score, so its support is known. The generator returns no value for the screen, so the code
computes it.

`law.expectation` integrates a function of the baseline covariates over the known law. A real
program cannot do that.


In [2]:
from cleverly.datasets import navigation_data, nonlinear_bounded_dgp

frame, truth = navigation_data(n=3_000, seed=31)
law = nonlinear_bounded_dgp()


def effect_of_offer(latent):
    return law.outcome_mean(latent, 1.0, None) - law.outcome_mean(latent, 0.0, None)


screen_truth = law.expectation(lambda latent: effect_of_offer(latent) * (latent[:, 0] > 0))
print("rows and columns:", frame.shape)
print(frame.head().round(3))
print()
print("known values of the synthetic law:")
print(f"  offer to all vs offer to none (the ATE): {truth['ate']:.3f}")
print(f"  screen on risk vs offer to none:         {screen_truth:.3f}")
print(f"  share of the ATE the screen keeps:       {screen_truth / truth['ate']:.3f}")

rows and columns: (3000, 6)
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             0.367                    1.0          -0.395              0.264              0.607 -0.972
1             0.768                    1.0           0.768              0.255              0.783  0.272
2             0.514                    0.0           1.162             -0.938              1.776  1.202
3             0.464                    1.0          -0.600              0.660              0.445 -1.746
4             0.775                    0.0           0.595             -0.585             -0.250 -0.602

known values of the synthetic law:
  offer to all vs offer to none (the ATE): 0.163
  screen on risk vs offer to none:         0.105
  share of the ATE the screen keeps:       0.648


**What this output tells you.** Each row is one discharge. `transition_navigation` is 1 for an
offer and 0 for usual support. The baseline covariates are standardized (mean 0, SD 1). The screen
offers navigation when `discharge_risk` is above 0, the population average.

| known value | what it means for the office |
| --- | --- |
| offer to all, 0.163 | the mean gain if every patient receives an offer. This is the ATE |
| screen on risk, 0.105 | the mean gain over all patients if only flagged patients receive an offer |
| share kept, 0.648 | the screen keeps about two thirds of the offer-to-all gain |

In this law the effect of an offer grows with `discharge_risk`, so the screen keeps most of the
benefit. Its value is still smaller than the ATE. The office needs that number to budget navigator
hours. No rescaling of the ATE produces it.

The next cell compares current practice with the screen.


In [3]:
flagged = frame["discharge_risk"] > 0
current = frame.groupby(flagged)["transition_navigation"].agg(["size", "mean"])
current.index = ["not flagged", "flagged"]
current.columns = ["patients", "share offered now"]
print(current.round(3))

             patients  share offered now
not flagged      1479              0.368
flagged          1521              0.586


**What this output tells you.** The screen flags 1521 patients and leaves 1479 unflagged. Current
practice already offers navigation more often to flagged patients, 0.586 against 0.368. Assignment
depends on discharge risk, so a raw comparison of the arms is confounded.
[Point-treatment TMLE](point-treatment-tmle.ipynb) shows that failure.

Both groups contain offered and unoffered patients. The screen asks for an offer in the flagged
group and for usual support in the other group. A regime needs positivity only for the arm it
assigns, so these shares give a first look at its support. Step 6 reads the support row by row.


## Axis 1: a known regime

A regime is a plan that assigns an arm from recorded baseline information. The office writes the
plan before fitting, so the data do not estimate it.

### Step 3: write the protocol

A `StudyProtocol` records the scientific design before any model runs. Each policy question on this
page gets its own protocol, because each question compares different strategies. This first
protocol starts from `navigation_protocol()`, the protocol of the
[shared study design](index.md#the-shared-study-design). The code uses `dataclasses.replace` to
list the three plans that the regime fit compares, and it adds one assumption rationale.

In [4]:
from cleverly.datasets import navigation_protocol

program = navigation_protocol()
regime_protocol = replace(
    program,
    treatment_strategies=(
        "Offer navigation to no eligible patient",
        "Offer navigation to every eligible patient",
        "Offer navigation when baseline discharge risk is above the population average",
    ),
    treatment_versions=(
        program.treatment_versions[1],
        program.treatment_versions[0],
        "The standard offer for a flagged patient, and usual support for every other patient",
    ),
    assumption_rationale=(
        *program.assumption_rationale,
        "The screen reads only discharge risk recorded before the offer",
    ),
)
print("\n".join(regime_protocol.summary_lines()))

causal study protocol: schema 1; 3f91dfe6affe6445
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer navigation to no eligible patient', 'Offer navigation to every eligible patient', 'Offer navigation when baseline discharge risk is above the population average']
treatment versions: ['No access to the transition-navigation offer', 'Bedside transition plan and two scheduled navigator contacts within 30 days', 'The standard offer for a flagged patient, and usual support for every other patient']
outcome: Patient-reported transition score, as a share of the maximum score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of 

**What this output tells you.** The first line gives the fingerprint `3f91dfe6affe6445`. The other
lines repeat each field. Every result fitted from this study carries the fingerprint.

The code changes three fields of the program protocol: the strategies, the versions, and the
assumption rationale.

| protocol field | what it records for the regime question |
| --- | --- |
| treatment strategies and versions, changed | the three plans: offer to none, offer to all, and the screen. Offer to none reuses the program's version for usual support |
| time zero, unchanged | the discharge-home order, before the offer, so the screen reads only baseline data |
| outcome, unchanged | the score as a share of the maximum, which Step 5 declares to the fit |
| assumption rationale, changed | the program's three arguments for the identification assumptions, and a fourth: the screen uses a pre-offer variable |

The protocol records the plans in words. It cannot store the rule itself, and it cannot check that
the `Rule` in the next step matches the words. The typed estimand owns the rule and the contrast. A
reviewer must compare the two.


### Step 4: design and identification

The design names the role of each column. The estimand names the three plans and the reference
plan. A `Static` plan assigns one arm to every patient. A `Rule` plan assigns an arm from a
function of the baseline data. The `identify` call returns the formula, the nuisances, and the
assumptions.

In [5]:
from cleverly import CausalStudy, PointTreatment, RegimeContrast
from cleverly.interventions import Rule, Static

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "prior_utilization", "medication_burden", "age"),
    ),
    protocol=regime_protocol,
)
plans = (
    Static(0, name="offer to none"),
    Static(1, name="offer to all"),
    Rule(lambda data: (data["discharge_risk"] > 0).astype(float), name="screen on risk"),
)
regimes = study.identify(RegimeContrast(plans, reference="offer to none"))
print(regimes.summary())

contrast of each regime against the reference
identified by explicit-adjustment: identified regime-indexed plug-in functional of E(transition_score | transition_navigation=a, W) for a in [0, 1] with influence correction
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity *for the regime*: g(a | W) > 0 wherever the regime assigns arm a with positive probability -- a weaker requirement than positivity for every arm when the regime is deterministic, and a different one
  - the regime is a known function of W: g* does not depend on the observed-data law, so the influence function carries no term for estimating it. An intervention defined *through* the estimated m

**What this output tells you.** The estimand is a contrast of each regime against the reference,
offer to none. The required nuisances are the outcome regression Q and the treatment mechanism g.
Two assumptions differ from those of the ATE.

| assumption | what it means for the screen |
| --- | --- |
| positivity for the regime | a patient needs some chance of the arm the plan assigns. A flagged patient needs some chance of an offer, and an unflagged patient needs some chance of usual support |
| the regime is a known function of W | the rule does not come from the data, so the influence curve needs no term for estimating it |

The influence curve records how much each row moves the estimate, and its variance gives the
standard error. [Inference](../technical-reference/inference.md) defines it. The summary ends with
the stored protocol and its fingerprint.


### Step 5: estimate the regime contrasts

The configuration is written out in full.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | gradient boosting | fits g, the probability of an offer given the covariates |
| `CrossFitting(n_folds=3)` | three folds | predicts each row from nuisance models fit without that row |
| `Targeting(q_bounds=(0.0, 1.0))` | the declared score support | fixes the outcome scale before any fold is drawn |
| `Runtime(random_state=31, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

[CV-TMLE](../technical-reference/cv-tmle.md) defines cross-fitting. A cross-fitted fit of a
continuous outcome must declare its support, because an undeclared scale is read from the held-out
rows as well. Step 3 recorded that support, so this fit states it. Axis 2 uses another law, whose
outcome has no such support, and Step 9 shows what that changes.

The code prints the result summary, then a table with the population value of each contrast.


In [6]:
from cleverly import CrossFitting, ModelSpec, Runtime, Targeting, TMLEMethod

method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=HistGradientBoostingRegressor(random_state=31),
        treatment_learner=HistGradientBoostingClassifier(random_state=31),
    ),
    cross_fitting=CrossFitting(n_folds=3),
    targeting=Targeting(q_bounds=(0.0, 1.0)),
    runtime=Runtime(random_state=31, n_jobs=1),
)
regime_result = regimes.estimate(method=method)
print(regime_result.summary())
print()
regime_truth = {
    "ate_regime[offer to all vs offer to none]": truth["ate"],
    "ate_regime[screen on risk vs offer to none]": screen_truth,
}


def against_truth(result, population_values):
    """Each estimate and its interval beside the population value of its estimand."""
    table = result.to_frame()[["estimand", "psi", "ci_lower", "ci_upper"]]
    table["population value"] = table["estimand"].map(population_values)
    return table


regime_table = against_truth(regime_result, regime_truth)
print(regime_table.round(3).to_string(index=False))

Targeted maximum likelihood estimation
n = 3000; covariates = 4; P(A=1) = 0.4787
causal estimand: contrast of each regime against the reference
identification: explicit-adjustment; identified regime-indexed plug-in functional of E(transition_score | transition_navigation=a, W) for a in [0, 1] with influence correction
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity *for the regime*: g(a | W) > 0 wherever the regime assigns arm a with positive probability -- a weaker requirement than positivity for every arm when the regime is deterministic, and a different one; the regime is a known function of W: g* does not depend on the observed-data law, so the influence function carries no term for estimating it. An intervention defined *through* the estima

**What this output tells you.** The summary names the construction as stacked CV-TMLE, with
nuisances cross-fitted over 3 folds. It truncates the propensity to [0.0114, 0.9886]. It also
carries the protocol fingerprint `3f91dfe6affe6445`.

| contrast | estimate | 95% interval | population value |
| --- | --- | --- | --- |
| offer to all vs offer to none | 0.172 | (0.145, 0.198) | 0.163 |
| screen on risk vs offer to none | 0.116 | (0.098, 0.135) | 0.105 |

Each interval contains its population value on this draw. One draw is not a coverage result. The
summary also prints simultaneous 95% bands, which cover both contrasts together. Their critical
value is 2.183 instead of 1.960.

The first contrast, offer to all against offer to none, is the ATE. Those two plans need support
in both arms.


### Step 6: the support of each plan

A rule needs positivity only where it assigns. Lower-risk patients receive usual support under the
screen, so the screen never divides by their probability of an offer. The support report publishes
one row for each plan.

The code also counts the rows whose observed arm differs from the arm that the screen assigns.


In [7]:
regime_assessment = regime_result.assess()
print("needs attention:", tuple(item.name for item in regime_assessment.attention))
print()
regime_support = regime_assessment.report("support")
print(regime_support.summary())
screen_load = regime_support.regimes["screen on risk"].score_load
off_plan = int((frame["transition_navigation"] != flagged.astype(float)).sum())
print()
print("score load of 'screen on risk':")
print(
    f"  Kish-equivalent rows:        {screen_load['effective']:.1f} of {screen_load['n_total']:.0f}"
)
print(f"  rows with zero load:         {screen_load['zero_load']:.0f}")
print(f"  rows whose arm is off-plan:  {off_plan}")

needs attention: ('nuisance_models',)

regime support (n = 3000)

regime          min g     max ratio  ratio effective n  score load               unsupported
--------------  --------  ---------  -----------------  -----------------------  -----------
offer to none   0.008104  82.94      476.5              476.6/3000 (draw 01/01)  0          
offer to all    0.001774  104        343.7              362.9/3000 (draw 01/01)  0          
screen on risk  0.006611  52.05      698.9              698.9/3000 (draw 01/01)  0          

score load is Kish-equivalent mask rows from abs(w_i * H_ij), not estimator ESS.

score load of 'screen on risk':
  Kish-equivalent rows:        698.9 of 3000
  rows with zero load:         1173
  rows whose arm is off-plan:  1173


**What this output tells you.** `needs attention` names `nuisance_models`, and Step 13 reads the
same warning on the incremental fit. The support table has one row for each plan.

| column | what it shows | the screen on this draw |
| --- | --- | --- |
| min g | the smallest fitted probability, over all rows, of the arm the plan assigns, before truncation | 0.006611 |
| max ratio | the largest ratio of the plan's probability to the fitted probability, at the observed arm, before truncation | 52.05 |
| ratio effective n | the effective sample size of those ratios | 698.9 of 3000 |
| score load | the effective sample size of the absolute weights in the plan's targeting equation | 698.9 of 3000 |

The first three columns use g before truncation. The score load uses the truncated g that the fit
used. The 0.0114 bound of Step 5 caps every weight at about 88. Offer to all has the largest ratio,
104, which is above that cap. Truncation therefore lifts its score load, 362.9, above its
ratio effective n, 343.7. The largest ratio of offer to none is 82.94, below the cap, so truncation barely moves
its two numbers: 476.5 against 476.6.

Offer to all has the smallest ratio effective n and the largest ratio, 104. It needs an offer
probability for every patient, including those whom current practice rarely offers navigation.

The screen assigns a different arm from the observed arm in 1173 rows. The score load of the screen
has 1173 rows with zero load, the same count. The targeting equation gives no weight to a row whose
observed arm differs from the plan.


## Axis 2: a modified treatment policy

The office sets a navigation intensity for each patient at discharge. A modified treatment policy
moves the intensity that each patient actually received. A continuous exposure needs a conditional
density rather than a propensity.

### Step 7: the intensity data

This policy uses its own synthetic law. The generator returns the population value of each declared
policy, so the code passes the four policies to it.


In [8]:
from cleverly.datasets import make_shift_dose

dose_frame, dose_truth = make_shift_dose(
    n=3_000,
    seed=32,
    shifts=(
        (0.0, None, "current practice"),
        (0.5, 5.0, "+0.5 capped at 5"),
        (0.5, None, "+0.5 uncapped"),
        (1.0, None, "+1.0 uncapped"),
    ),
)
dose_frame = dose_frame.rename(
    columns={
        "Y": "transition_score",
        "A": "assigned_navigation_intensity",
        "W1": "discharge_risk",
        "W2": "navigator_caseload",
        "W3": "caregiver_support",
    }
)
intensity = dose_frame["assigned_navigation_intensity"]
print("rows and columns:", dose_frame.shape)
print(dose_frame.head().round(3))
print()
print(
    f"assigned intensity: mean {intensity.mean():.3f}, minimum {intensity.min():.3f}, "
    f"maximum {intensity.max():.3f}, share below zero {np.mean(intensity < 0):.1%}"
)
print()
print("known values of the synthetic law:")
for name, value in dose_truth.items():
    print(f"  {name:50s} {value:.4f}")

rows and columns: (3000, 5)
   transition_score  assigned_navigation_intensity  discharge_risk  navigator_caseload  caregiver_support
0             1.694                          0.358           0.909               0.751             -0.026
1             5.417                          2.977           0.653              -0.651             -1.829
2             4.794                          2.306           1.679               0.477             -2.066
3             0.334                          1.579          -1.059              -0.620              0.342
4             2.644                          1.082           1.327               0.308             -0.433

assigned intensity: mean 1.978, minimum -2.741, maximum 7.113, share below zero 5.1%

known values of the synthetic law:
  ey_shift[current practice]                         3.3950
  ey_shift[+0.5 capped at 5]                         4.1711
  ey_shift[+0.5 uncapped]                            4.2075
  ey_shift[+1.0 uncapped]         

**What this output tells you.** Each row is one discharge, and `assigned_navigation_intensity` is
the continuous exposure. The intensity is in synthetic units with no physical zero, so 5.1% of the
values are below zero. The mean intensity is 1.978. Given the covariates, the intensity is normal
with SD 1, as `shift_dgp` in `src/cleverly/datasets/synthetic.py` declares.

The known values give each policy mean, `ey_shift`, and each contrast against current practice,
`ate_shift`. The two `+0.5` contrasts apply the same increase and have different values, 0.7761 and
0.8125. Step 10 explains the gap.

### Step 8: write the protocol, then identify the policies

The intensity question keeps the population, the eligibility, and the horizon of the program
protocol. It changes the outcome, the time zero, the strategies, the versions, the
intercurrent-event handling, and the assumption rationale. `dataclasses.replace` copies `program`
and changes only those fields.

The outcome field matters for the fit. This law records a standardized score, which takes its scale
from the data, so no finite support can be declared for it. Step 9 returns to that.

`treatment_kind="continuous"` tells the design that the exposure is continuous.
`Shift(delta, cap=..., name=...)` adds `delta` to the intensity of each patient. A patient whose
new intensity would exceed the cap keeps the current intensity. `cap` has no default, so every
policy states it, and `cap=None` declares no cap.


In [9]:
from cleverly import ModifiedTreatmentPolicyEffect
from cleverly.interventions import Shift

dose_protocol = replace(
    program,
    outcome="Patient-reported transition score, standardized to the baseline distribution",
    time_zero=(
        "Discharge-home order, after baseline measurement and before the navigation "
        "intensity is assigned"
    ),
    treatment_strategies=(
        "Keep the navigation intensity that current practice assigns",
        "Raise the assigned intensity by 0.5 units unless the result would exceed the "
        "declared capacity of 5 units",
        "Raise the assigned intensity by 0.5 units",
        "Raise the assigned intensity by 1.0 unit",
    ),
    treatment_versions=(
        "Navigator contacts at the intensity current practice assigns",
        "Navigator contacts at the raised intensity, or at the current intensity where "
        "the raise would exceed 5 units",
        "Navigator contacts at the raised intensity",
        "Navigator contacts at the raised intensity",
    ),
    intercurrent_event_handling=(
        program.intercurrent_event_handling[0],
        "Analyze the assigned intensity regardless of the contacts delivered",
        program.intercurrent_event_handling[2],
    ),
    assumption_rationale=(
        "Discharge risk, navigator caseload, and caregiver support cover the measured "
        "common causes",
        "Intensity records in navigator contact units support consistency",
        "The staffing plan sets the capacity of 5 units before fitting, not these data",
        program.assumption_rationale[2],
    ),
)
dose_study = CausalStudy(
    dose_frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="assigned_navigation_intensity",
        adjustment=("discharge_risk", "navigator_caseload", "caregiver_support"),
        treatment_kind="continuous",
    ),
    protocol=dose_protocol,
)
policies = (
    Shift(0.0, cap=None, name="current practice"),
    Shift(0.5, cap=5.0, name="+0.5 capped at 5"),
    Shift(0.5, cap=None, name="+0.5 uncapped"),
    Shift(1.0, cap=None, name="+1.0 uncapped"),
)
shift_effect = dose_study.identify(ModifiedTreatmentPolicyEffect(policies))
print(shift_effect.summary())

contrast of modified treatment policies
identified by explicit-adjustment: identified shift-indexed plug-in functional of E(transition_score | assigned_navigation_intensity=a, W) with influence correction
adjustment/history: ['discharge_risk', 'navigator_caseload', 'caregiver_support']
required nuisances: ['outcome_regression', 'treatment_density']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity *for the shifted dose*: g(d(a, w) | w) > 0 wherever g(a | w) > 0, so the dose the policy assigns is one the data have seen at that covariate value. This is weaker than positivity at every dose -- which no continuum satisfies -- and is exactly what the cap is declared to secure
  - the shift is a known function of (A, W): d does not depend on the observed-data law, so the influence function carries no term for estim

**What this output tells you.** The protocol fingerprint is `31d1fe08b6dc70ac`. It differs from
Step 3, because the fields differ. The four strategies name the policies in the order of the fit.

The capped strategy states the capacity of 5 units in words. The protocol has no numeric field for
the cap, and it cannot check the `cap=5.0` in the code. The rationale records that the staffing plan
sets the cap before fitting. The second assumption in the next table needs that record.

The required nuisances are the outcome regression and `treatment_density`, the conditional density
of the intensity. Two assumptions are new.

| assumption | what it means for the program |
| --- | --- |
| positivity for the shifted dose | the data must contain the intensity a policy assigns, at that covariate value |
| the shift is a known function of (A, W) | the cap comes from the protocol, not from the data. A cap fitted from the data would define a different parameter |

The summary also lists assumptions for `delta=` and `intermediate=`. This fit declares neither, so
those two do not apply.


### Step 9: estimate the policy contrasts

The fit declares four policies. The first changes nothing, so its population mean is the mean under
current practice. The other three apply an increase under different statements about what staffing
can deliver.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the intensity and the covariates |
| `treatment_learner` | gradient boosting classifier | fits the conditional density of the intensity as a pooled hazard over bins |
| `density_bins=40` | 40 bins | sets the resolution of that density |
| `CrossFitting(enabled=False)` | no outer folds | fits each nuisance on every row |

This axis fits in sample, and Step 8 gives the reason. The outcome here is a standardized score
with no known finite support, so a cross-fitted fit could declare no `q_bounds`, and `cleverly`
refuses that fit. The registered study for this axis also targets in sample. The reading below
states what the in-sample fit costs on this draw.

The fit can raise a `PositivityWarning`. The code records each warning and prints it above the
results.


In [10]:
from cleverly import PositivityWarning

shift_method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=HistGradientBoostingRegressor(random_state=32),
        treatment_learner=HistGradientBoostingClassifier(random_state=32),
        density_bins=40,
    ),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=32, n_jobs=1),
)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", PositivityWarning)
    shift_result = shift_effect.estimate(method=shift_method)
positivity_warnings = [str(w.message) for w in caught if w.category is PositivityWarning]
print("\n".join(positivity_warnings))
print()
shift_table = against_truth(shift_result, dose_truth)
shift_table["interval width"] = shift_table["ci_upper"] - shift_table["ci_lower"]
print(shift_table.round(3).to_string(index=False))
print()
print("protocol fingerprint on the result:", shift_result.provenance.protocol_fingerprint)

shift '+0.5 uncapped' has cap=None, and 0.1% of rows are assigned a dose above the largest one observed (7.11). The outcome regression is extrapolating there, and identification needs the shifted dose to be supported. Declare a cap= if you know what dose is achievable.
shift '+1.0 uncapped' has cap=None, and 0.1% of rows are assigned a dose above the largest one observed (7.11). The outcome regression is extrapolating there, and identification needs the shifted dose to be supported. Declare a cap= if you know what dose is achievable.

                                       estimand   psi  ci_lower  ci_upper  population value  interval width
ate_shift[+0.5 capped at 5 vs current practice] 0.788     0.765     0.811             0.776           0.046
   ate_shift[+0.5 uncapped vs current practice] 0.825     0.802     0.848             0.812           0.046
   ate_shift[+1.0 uncapped vs current practice] 1.800     1.752     1.848             1.750           0.096

protocol fingerprint on th

**What this output tells you.** Two warnings name the uncapped policies. Some rows receive an
intensity above the observed maximum of 7.11, and the outcome regression extrapolates there. The
capped policy raises no warning.

| policy against current practice | estimate | 95% interval | population value | interval width |
| --- | --- | --- | --- | --- |
| `+0.5 capped at 5` | 0.788 | (0.765, 0.811) | 0.776 | 0.046 |
| `+0.5 uncapped` | 0.825 | (0.802, 0.848) | 0.812 | 0.046 |
| `+1.0 uncapped` | 1.800 | (1.752, 1.848) | 1.750 | 0.096 |

On this draw, the `+1.0 uncapped` interval is about twice as wide as either `+0.5` interval. It is
also the one interval that excludes its population value, by a small margin. Two facts about this
fit bear on that miss.

The nuisances are fitted in sample, so each row's prediction comes from a
model that saw it. The [cross-fitting tutorial](cross-fitting.ipynb) shows that such an interval
can be too narrow. The `+1.0` shift also carries the most positivity strain, which Step 11
measures. One draw is not a coverage result either way.

The result carries the protocol fingerprint `31d1fe08b6dc70ac`. Step 11 reads the support behind
the wider interval.


### Step 10: the failure mode, the cap is part of the question

The two `+0.5` policies apply the same increase. The declared cap holds back each patient whose
new intensity would exceed 5. Those patients keep their current intensity under one policy and not
under the other. The code measures how much that difference matters on this draw.

In [11]:
dose = intensity.to_numpy()
capped_name = "ate_shift[+0.5 capped at 5 vs current practice]"
uncapped_name = "ate_shift[+0.5 uncapped vs current practice]"
capped, uncapped = shift_result[capped_name], shift_result[uncapped_name]
gap = shift_result.contrast(
    lambda p: p[0] - p[1], [uncapped_name, capped_name], name="uncapped - capped"
)
low, high = gap.ci
print(f"rows the cap holds back:              {np.mean(dose + 0.5 > 5.0):.1%}")
print(
    "population value, uncapped - capped: "
    f"{dose_truth[uncapped_name] - dose_truth[capped_name]:.3f}"
)
print(f"estimate, uncapped - capped:          {gap.psi:.3f} (95% CI {low:.3f} to {high:.3f})")
print(f"standard error of the gap:            {gap.std_error:.4f}")
print(
    f"standard error of each estimate:      {capped.std_error:.4f} (capped), "
    f"{uncapped.std_error:.4f} (uncapped)"
)
print()
for delta in (0.5, 1.0):
    beyond = int(np.sum(dose + delta > dose.max()))
    print(f"rows shifted above the observed maximum by +{delta}: {beyond}")

rows the cap holds back:              2.5%
population value, uncapped - capped: 0.036
estimate, uncapped - capped:          0.037 (95% CI 0.026 to 0.047)
standard error of the gap:            0.0054
standard error of each estimate:      0.0118 (capped), 0.0116 (uncapped)

rows shifted above the observed maximum by +0.5: 2
rows shifted above the observed maximum by +1.0: 3


**What this output tells you.** The cap holds back 2.5% of the rows. The population values of the
two policies differ by 0.036. The estimated gap is 0.037, with a 95% interval of (0.026, 0.047). On
this draw, the interval contains the population gap and excludes zero.

`contrast` computes the gap from the joint influence curve, because both estimates come from the
same rows. The two policies differ only on the rows that the cap holds back. The standard error of
the gap is therefore 0.0054, below the standard error of each estimate, 0.0118 and 0.0116. Do
not judge a gap between two estimates of one fit against the standard error of one estimate.

The gap is small because few rows reach the cap. That is a fact about these data, not about the
estimand. A staffing ceiling inside the bulk of the distribution would separate the two policies
further.

`cap` has no default, so the office must state what is achievable. That is a question about the
world. The
[modified treatment policies](../technical-reference/point-treatment-tmle.md#modified-treatment-policies)
section explains why a cap estimated from the data would change the parameter.

The last two lines count the rows that an uncapped shift moves above the observed maximum. The
count is 2 for `+0.5` and 3 for `+1.0`. The warnings in Step 9 report these rows.


### Step 11: the support of each shift

The support report of a shift fit has one entry for each policy. Each entry gives the density
ratio of that shift and its score load.

The density ratio omits observation weights. The score load includes the exact weights and score
mask of the fit. Read both quantities, because either one can be the more concentrated.

In [12]:
shift_assessment = shift_result.assess()
print("needs attention:", tuple(item.name for item in shift_assessment.attention))
print()
for report in shift_assessment.report("support").values():
    print(report.summary())
for delta in (0.5, 1.0):
    print(f"Kish fraction under the true density ratio for +{delta}: {np.exp(-(delta**2)):.1%}")

needs attention: ()

current practice: min g(A|W)=0.00246, max ratio=1, ESS=3000 (100.0% of n), capped=0.0%, unsupported=0, score load=3000.0/3000 Kish-equivalent mask rows (draw 01 of 01)
    ratio quantiles -- 1%: 1, 5%: 1, 50%: 1, 95%: 1, 99%: 1
+0.5 capped at 5: min g(A|W)=0.00246, max ratio=8.48, ESS=1858 (61.9% of n), capped=2.5%, unsupported=0, score load=1857.7/3000 Kish-equivalent mask rows (draw 01 of 01)
    ratio quantiles -- 1%: 0.133, 5%: 0.257, 50%: 0.831, 95%: 2.17, 99%: 4.13
+0.5 uncapped: min g(A|W)=0.00246, max ratio=8.27, ESS=1952 (65.1% of n), capped=0.0%, unsupported=0, score load=1952.0/3000 Kish-equivalent mask rows (draw 01 of 01)
    ratio quantiles -- 1%: 0.133, 5%: 0.257, 50%: 0.831, 95%: 2.12, 99%: 3.85
+1.0 uncapped: min g(A|W)=0.00246, max ratio=51.3, ESS=733 (24.4% of n), capped=0.0%, unsupported=0, score load=733.2/3000 Kish-equivalent mask rows (draw 01 of 01)
    ratio quantiles -- 1%: 0.0382, 5%: 0.0785, 50%: 0.581, 95%: 3.03, 99%: 6.21
Kish fraction

**What this output tells you.** `needs attention` is empty, but the support differs sharply between
the policies. An empty list is not evidence of support.

| policy | what the report shows on this draw | how to read it |
| --- | --- | --- |
| current practice | the maximum ratio is 1, and the effective sample size is the full sample of 3000 rows | nothing moved, so nothing was extrapolated |
| `+0.5 capped at 5` | 2.5% of rows capped, and an effective sample size of 61.9% of the rows | the ratio moves weight onto some rows, so these 3000 rows carry the precision of about 1858 equally weighted rows |
| `+1.0 uncapped` | an effective sample size of 24.4% of the rows, and a maximum ratio of 51.3 | the estimated ratio is concentrated on a few patients, and the estimate rests on them |

This fit declares no observation weights, so each effective sample size and its score load agree
closely.

Intensity is normal with SD 1 given the covariates, so every shift is identified in the population.
With the true density, the Kish fraction of a shift of size δ is exp(-δ²). That fraction is 77.9%
for a shift of 0.5 and 36.8% for a shift of 1.0. The estimated 40-bin density gives 65.1% for
`+0.5 uncapped` and 24.4% for `+1.0 uncapped`. That gap is practical positivity strain in a finite
sample. On this draw, the strain is much larger for `+1.0`.

The wider `+1.0` interval in Step 9 is consistent with the strain. The response is quadratic in
intensity, so the `+1.0` shift also has a larger effect, and it extrapolates for some rows. Under
strain, the influence-curve standard error can itself be too small. A larger sample, a smaller
shift, or a declared cap reduces the strain.


## Axis 3: an incremental intervention

The third proposal is not one fixed assignment. The program multiplies the current conditional odds
of an offer for each patient by two. This stochastic policy is the intervention.

### Step 12: write the protocol, then identify the policy

The incremental question uses the data of Step 2. Its protocol changes the strategies, the
versions, and the assumption rationale of the program protocol.

The design is the same as in Step 4, so the code reuses `study.design`. The policy
`Incremental(2.0)` doubles the odds, and `Incremental(1.0)` keeps the current odds.
`law.incremental_truth` returns the population values of the known law.

In [13]:
from cleverly import IncrementalEffect
from cleverly.interventions import Incremental

incremental_protocol = replace(
    program,
    treatment_strategies=(
        "Keep each patient's current conditional odds of a navigation offer",
        "Multiply each patient's current conditional odds of a navigation offer by two",
    ),
    treatment_versions=(
        "The offer as the current discharge-team process assigns it",
        "The standard offer, assigned at random with the doubled odds given the recorded "
        "baseline variables",
    ),
    assumption_rationale=(
        *program.assumption_rationale,
        "The program can run a randomized offer at the doubled odds",
    ),
)
incremental_truth = law.incremental_truth((1.0, 2.0))
incremental_study = CausalStudy(frame, design=study.design, protocol=incremental_protocol)
incremental_effect = incremental_study.identify(
    IncrementalEffect(
        (
            Incremental(1.0, name="current odds"),
            Incremental(2.0, name="double odds"),
        )
    )
)
print(incremental_effect.summary())

contrast of incremental propensity interventions
identified by explicit-adjustment: identified ipsi-indexed plug-in functional of E(transition_score | transition_navigation=a, W) for a in [0, 1] with influence correction
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - *no positivity assumption*: the clever covariate is delta/D at A=1 and 1/D at A=0 with D = delta*g + 1 - g, so it lies between min(delta, 1/delta) and max(delta, 1/delta) however small g is. This is the estimand's reason for existing, not an oversight in the list
  - the intervention is a functional of the observed-data law: q_delta is built out of g, so the influence function carries a term for the pa

**What this output tells you.** The protocol fingerprint is `17aa1c666565ec98`. The two strategies
keep the current odds or double them. The strategies depend on the current assignment process. The
protocol has no field that fixes that process, because the observed mechanism defines it.

The identification summary states the trade in two assumptions.

| assumption | what it means for the program |
| --- | --- |
| no positivity assumption | the outcome-targeting score weight stays between one half and two for this multiplier, however small g is |
| the intervention is a functional of the observed-data law | the policy is built from g, so the estimator also targets the mechanism |

Double robustness means that the point estimate stays consistent when either nuisance model is
consistent. The mechanism defines this estimand, so the estimate is not double robust. A good
outcome regression cannot rescue an inconsistent mechanism estimate.
[Kennedy (2019)](../references.md#incremental-interventions) derives both properties, and the
[incremental section](../technical-reference/point-treatment-tmle.md#incremental-propensity-score-interventions)
states them.


### Step 13: estimate the incremental contrast

The fit reuses the `method` object from Step 5, so only the estimand and the protocol change. The
code prints the estimate beside the population value, and its distance from that value in
standard errors. It then prints the nuisance report and the
score-equation report of the assessment.

In [14]:
incremental_result = incremental_effect.estimate(method=method)
double_odds = incremental_truth["ate_ipsi[odds x2 vs natural course]"]
incremental_table = against_truth(
    incremental_result, {"ate_ipsi[double odds vs current odds]": double_odds}
)
print(incremental_table.round(3).to_string(index=False))
tilt = incremental_result["ate_ipsi[double odds vs current odds]"]
distance = (tilt.psi - double_odds) / tilt.std_error
print(f"distance from the population value: {distance:.1f} standard errors")
print("protocol fingerprint on the result:", incremental_result.provenance.protocol_fingerprint)
print()
incremental_assessment = incremental_result.assess()
print("needs attention:", tuple(item.name for item in incremental_assessment.attention))
print()
print(incremental_assessment.report("nuisance_models").summary())
print()
print(incremental_assessment.report("score_equations").summary())

                             estimand   psi  ci_lower  ci_upper  population value
ate_ipsi[double odds vs current odds] 0.026     0.024     0.028             0.023
distance from the population value: 2.6 standard errors
protocol fingerprint on the result: 17aa1c666565ec98

needs attention: ('nuisance_models',)

Nuisance model diagnostics
----------------------------------------
model       auc     brier   log_loss  r2      mse     cal_slope
----------  ------  ------  --------  ------  ------  ---------
propensity  0.6766  0.2388  0.6892    -       -       0.4584   
outcome     -       -       -         0.6490  0.0201  0.9525   

VERDICT:
  - propensity is poorly calibrated (slope 0.46, ideal 1.0); its predicted probabilities are systematically off, which biases the weights

Score-equation check
--------------------
target                                 kind             |score|    before     threshold  ratio     ok 
-------------------------------------  ---------------  ---------  --

**What this output tells you.** The estimate is 0.026, with a 95% interval of (0.024, 0.028). The
population value is 0.023, so the interval excludes it on this draw. The estimate lies 2.6
standard errors above the population value. A 95% interval misses on about one draw in 20, so one
miss is consistent with sampling variation.

`needs attention` names `nuisance_models`. The boosted propensity is poorly calibrated, with a
calibration slope of 0.46 against an ideal of 1. A slope below 1 means that the predicted
probabilities are too extreme.

The regime fit in Step 6 carries the same warning. The regime estimate is double robust, and
Step 12 explains why the incremental estimate is not. One draw cannot show whether the miss comes
from g. Try a better-calibrated treatment learner before you report this axis.

The score-equation check has two fluctuation rows, `ipsi` and `ipsi (mechanism)`. The second row is
the targeting equation for the treatment mechanism. No other axis on this page has that equation.
The check marks both rows as solved.

The next cell prints the support report for each odds multiplier.


In [15]:
for report in incremental_assessment.report("support").values():
    print(report.summary())

current odds: min g(1|W)=0.00177, covariate in [1, 1] by construction, max=1, ESS=3000 (100.0% of n), score load=3000.0/3000 Kish-equivalent mask rows (draw 01 of 01)
double odds: min g(1|W)=0.00177, covariate in [0.5, 2] by construction, max=1.98, ESS=2692 (89.7% of n), score load=2711.0/3000 Kish-equivalent mask rows (draw 01 of 01)


**What this output tells you.** The double-odds outcome score weight lies in [0.5, 2]. Its
largest value is 1.98. The effective sample size is 89.7% of
the rows. This factor gives no row more than twice unit load in that equation. The `+1.0` shift in
Step 11 kept 24.4% on its own data.

This score load describes the outcome targeting equation for each odds multiplier. The incremental
fit also targets the treatment mechanism. That second equation has no row-level load in this
report.


## Step 14: three tables that look alike, and their assessments

Each `to_frame()` table above has the same columns. The code prints the estimand names of each fit.
A combined assessment presents validation, diagnostics, and sensitivity for one fit. The code then
prints the status of each operation, with one column for each axis.

In [16]:
axes = {
    "known regime": (regime_result, regime_assessment),
    "modified treatment policy": (shift_result, shift_assessment),
    "incremental intervention": (incremental_result, incremental_assessment),
}
for axis, (fitted, _) in axes.items():
    print(f"{axis}: {list(fitted.to_frame()['estimand'])}")
print()
statuses = {}
for axis, (_, assessment) in axes.items():
    ledger = assessment.to_frame()
    statuses[axis] = ledger.set_index(ledger["surface"] + "." + ledger["check"])["status"]
status = pd.DataFrame(statuses)
print(status.to_string())

known regime: ['ate_regime[offer to all vs offer to none]', 'ate_regime[screen on risk vs offer to none]']
modified treatment policy: ['ate_shift[+0.5 capped at 5 vs current practice]', 'ate_shift[+0.5 uncapped vs current practice]', 'ate_shift[+1.0 uncapped vs current practice]']
incremental intervention: ['ate_ipsi[double odds vs current odds]']

                                     known regime modified treatment policy incremental intervention
validation.score_equations                 passed                    passed                   passed
validation.support                      completed                 completed                completed
validation.nuisance_models                warning                 completed                  warning
diagnostics.corrections            not_applicable            not_applicable           not_applicable
diagnostics.truncation_curve             deferred                  deferred                 deferred
diagnostics.refute                       de

**What this output tells you.** The `estimand` column separates the three tables: `ate_regime`,
`ate_shift`, and `ate_ipsi`. Carry that name into the program report.

The library fits one intervention axis at a time. It has no joint targeting and no joint
influence-curve covariance for two axes. The intensity policy also uses a different treatment
column. Fit each policy question separately.

The status table has one row for each operation of the combined assessment.

| status | what it means for these fits |
| --- | --- |
| `passed` | each axis solved its score equations |
| `completed` | the operation returned a report with no verdict. The support reports of Steps 6, 11, and 13 are `completed` for all three axes |
| `warning` | the propensity is poorly calibrated. The regime fit and the incremental fit use the same data and the same treatment learner, so both warn |
| `deferred` | the operation refits or retargets, so it runs only on request. Step 15 runs `simulated_confounding` on the regime fit |
| `unavailable` | the omitted-variable bounds. Step 15 gives the reason |
| `not_applicable` | the operation does not apply to these fits |

For the shift fit, `nuisance_models` is `completed`. The nuisance report is returned, but it gives
no calibration verdict. Read the support report of each axis before you compare its estimates.

## Step 15: sensitivity, what the fits cannot check

Diagnostics cannot see unmeasured confounding. A
[sensitivity analysis](../user-guide/results-assessment.md#sensitivity-analysis) asks how strong an
unmeasured confounder would need to be to change the conclusion. The code first asks for the
robustness value of the regime fit. It then runs a simulated common-cause stress surface for the
screen contrast.

| setting | value | what it does |
| --- | --- | --- |
| `treatment=(0.0, 0.1)` | flip strengths | flips the treatment of each row whose latent value is in the top 10% of the standard normal |
| `outcome=(0.0, 0.5)` | Gaussian strengths | subtracts the strength times the latent value from the outcome |

Each nonzero cell refits both nuisances, so this surface costs three refits.

In [17]:
from statistics import NormalDist

from cleverly import CapabilityError
from cleverly.sensitivity import ConfounderStrengthGrid

try:
    regime_result.sensitivity.robustness_value()
except CapabilityError as refusal:
    print("robustness value refused:", refusal)
print()
surface = regime_result.sensitivity.simulated_confounding(
    estimand="ate_regime[screen on risk vs offer to none]",
    grid=ConfounderStrengthGrid(treatment=(0.0, 0.1), outcome=(0.0, 0.5)),
)
latent = np.random.default_rng(surface.latent_seed).normal(size=len(frame))
flip_mask = latent >= NormalDist().inv_cdf(0.9)
original_treatment = frame["transition_navigation"].to_numpy()
flipped_treatment = np.where(flip_mask, 1.0 - original_treatment, original_treatment)
offer_rates = {
    "before flips": float(original_treatment.mean()),
    "after flips": float(flipped_treatment.mean()),
}
print(f"offer rate before flips: {offer_rates['before flips']:.4f}")
print(f"offer rate after flips:  {offer_rates['after flips']:.4f}")
print(f"offer-rate change:       {offer_rates['after flips'] - offer_rates['before flips']:+.4f}")
print()
print(surface.summary())
print()
for failure in surface.failures[:1]:
    print(f"why each outcome-strength cell failed ({failure.error_type}): {failure.message}")

robustness value refused: the omitted-variable bound applies to the arm-indexed linear estimands, and this fit reports none: its parameters are indexed by 'regime' and it reported ['ate_regime[offer to all vs offer to none]', 'ate_regime[screen on risk vs offer to none]']. The bias is bounded through the Riesz representer of a mean or a contrast of arms, which a fit whose counterfactuals are not arms does not have.



offer rate before flips: 0.4787
offer rate after flips:  0.4910
offer-rate change:       +0.0123

Simulated common-cause stress surface for 'ate_regime[screen on risk vs offer to none]'
original estimate: 0.11645
seeds: root=31, latent=2459163703, refit=31
cross-fitting: 1 draw, no repeat aggregation
target measure: unweighted
target population: baseline
conditioning arm: None
baseline stratum: None
strata columns: ()
association population: full_fitted_population
calibration population: full_fitted_population
refit population: full_fitted_population
Treatment is flipped in the declared upper latent-normal tail.
Gaussian outcome subtracts signed strength times the shared latent value.

treatment strength  outcome strength  estimate  movement (estimate difference)  induced association  population fraction
------------------  ----------------  --------  ------------------------------  -------------------  -------------------
0                   0                 0.11645   +0             

**What this output tells you.** The robustness value is refused. The implemented omitted-variable
bound covers only arm-indexed estimands, such as an arm mean or the ATE. The status
table in Step 14 lists the same bound as `unavailable` for all three axes.

The message says that a regime fit has no Riesz representer. A deterministic regime mean has one:
the ratio that Step 6 reports. The bound of Chernozhukov et al. (2022) also covers a regime mean,
but the library does not implement that case.
[Sensitivity analysis](../references.md#sensitivity-analysis) cites the paper.

The
[stress-surface section](../technical-reference/validation-methods.md#simulated-common-cause-stress-surface)
lists regime, shift, and incremental parameters among the targets of the simulated surface. Two of
its four cells return an estimate, and two record a failure.

| treatment strength | outcome strength | estimate | movement | reading |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0.11645 | 0 | the original fit |
| 0 | 0.5 | failed | none | the perturbed score leaves the support the fit declares |
| 0.1 | 0 | 0.086852 | -0.029598 | the latent value moves the treatment only. The flips misclassify treatment |
| 0.1 | 0.5 | failed | none | the same failure, with the flips applied as well |

The flipped cell that ran moves the estimate by -0.029598, and its induced association is +0.0442.
The summary reads that movement as misclassification of the treatment, not as confounding. A flip
turns an offer into usual support, and usual support into an offer. The offer rate rises from
0.4787 to 0.4910, a change of +0.0123. The swaps go in both directions, but they do not cancel
exactly.

The last line gives the reason both outcome-strength cells failed. The surface subtracts the
strength times a standard normal latent value from the score. That sends the perturbed score to
a range of about -1.35 to 2.28. The fit declares the support (0.0, 1.0), so its refit refuses those
values.

A declared support is a statement about the measurement, and this perturbation does not
keep the measurement. This version of `cleverly` has no outcome perturbation on the score's own
scale, so the outcome axis of the surface is unavailable whenever a fit declares `q_bounds`. The
surface records each failure per cell rather than abandoning the run.

This grid therefore shows one treatment-side movement and no outcome-side movement. It does not
show how strong a confounder would need to be to change the screen conclusion. The surface is
qualitative. It is not a bound, and it is not an adjusted interval. Choose strengths that the
causal review of the program can defend before you read a movement.


## How far to trust this

| layer | establishes | does not establish |
| --- | --- | --- |
| the combined assessments | which checks passed, warned, completed, or were omitted for each axis | that one axis answers the policy question of another axis |
| the per-policy support reports | how concentrated the weights of each declared policy are on these data | that a well-supported policy is worth adopting |
| the score-equation checks | that each axis solved its own score equations | that the chosen axis matches the decision the office faces |
| the simulated stress surface | how the screen estimate moves under one declared latent common cause on the treatment side | a bound, that no such confounder exists, or any outcome-side movement, which this fit's declared support refuses |

These fits do not validate the estimators. The
[validation grid](../technical-reference/method-evidence/validation-grid.md) has one registered
study for each axis.

| axis | registered study | what the study does not cover here |
| --- | --- | --- |
| known regime | [deterministic point-treatment regimes](../technical-reference/method-evidence/deterministic-point-treatment-regimes.md) | ordinary, non-cross-fitted targeting, pointwise intervals only, a binary outcome, and no flexible learners |
| modified treatment policy | [continuous modified treatment policies](../technical-reference/method-evidence/continuous-modified-treatment-policies.md) | a 320-bin density, shifts of +0.25 and a capped +0.5 only, and no flexible learning |
| incremental intervention | [incremental propensity-score interventions](../technical-reference/method-evidence/incremental-propensity-interventions.md) | ordinary, non-cross-fitted targeting, a binary outcome, and no flexible learners |

No registered study covers the boosted learners, the three-fold cross-fitting, or the 40-bin
density used here. The intensity axis fits its boosted learners in sample, so its interval also
needs the data-reuse condition that
[CV-TMLE](../technical-reference/cv-tmle.md#what-this-solves) states.

Choose the axis that represents a change the program can implement. Then use its support report to
decide whether these data can estimate that policy well enough to report.


## Where to go next

Every axis on this page acts at one decision time. Read [longitudinal TMLE](longitudinal-tmle.ipynb)
for a rule that assigns navigation again at day seven. Read [MSM projections](msm-projections.ipynb)
for the working model that summarizes many plans as one trend. These three axes replace the average
treatment effect, and [point-treatment TMLE](point-treatment-tmle.ipynb) estimates it.

The [examples index](index.md#the-program) lists every tutorial in the program.
